In [ ]:
import pandas as pd 
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt 
from PIL import Image
from pathlib import Path

In [ ]:
lab_flat_fields = {'T': 'lab_flat_field_target.fits', 
                   'G_OLD': 'lab_flat_field_global_old.fits', 
                   'G': 'lab_flat_field_global.fits'}  
cal_reference = 'obs_cal_info.csv'
cal_channel_and_sample_indices = {
    # col 0 also affected but they don't interpolate it bc it's in the dark area 
    'detector_array_tap_columns': {
        'T': [160, 320, 480],
        'G': [80, 160, 240]
    },
    'filter_seam_channels': {
        'T': [40, 41, 115],
        'G': [12, 49]
    },
    'masked_columns_split': {
        'TL': [0, 1, 2, 3, 4, 5, 6, 7],
        'TR': [636, 637, 638, 639],
        'GL': [0, 1, 2, 3], # could do one less? they don't say. 
        'GR': [318, 319]
    },
    'masked_columns': {
        'T': [0, 1, 2, 3, 4, 5, 6, 7, 636, 637, 638, 639],
        'G': [0, 1, 2, 3, 318, 319]
    },
    'scattered_light_columns_split': {
        'TL': [8, 9, 10, 11, 12, 13, 14],
        'TR': [627, 628, 629, 630, 631, 632, 633, 634, 635],
        'GL': [4, 5, 6, 7],
        'GR': [314, 315, 316, 317, 318]
    },
    
    'scattered_light_columns': {
        'T': [8, 9, 10, 11, 12, 13, 14, 627, 628, 629, 630, 631,
              632, 633, 634, 635],
        'G': [4, 5, 6, 7, 314, 315, 316, 317, 318],
    },
    'channels_omitted_at_level1b': {
        'T': [0],
        'G': [0, 1, 2, 3]
    },
}



In [ ]:
def make_dark_signal_mean_image(dark_l0_path: str): 
    """
    The dark signal of an observation is estimated from a dark signal observation
    acquired prior to the real observation during a non-illuminated portion of
    the orbit. They were also used to generate the bad detector element image 
    (BDE), but we are not currently doing that. 

    The dark signal is averaged for all "lines" to get an average dark signal
    value for each cross-track sample and spectral channel. 
    """
    from astropy.io import fits

    with fits.open(dark_l0_path) as hdul:
        dark_obs_data = hdul[0].data

    # check everything looks normal (it should) 
    bands, rows, cols = dark_obs_data.shape

    if rows <= 4: 
        print("This dark signal obs is probably too short to be useful.")
        
    if bands not in (86, 260) or cols not in (320, 640):
        print("This has the wrong number of channels and/or spatial samples.") 
        
    # this is not listed in the DPSIS or Green but I think it makes sense to
    # exclude the first two frames and the last two when computing the mean 
    # dark signal bc sometimes they exhibit odd and anomalous characteristics.
    dark_signal_avg = dark_obs_data.transpose(1, 0, 2)[2:-2, :, :].mean(axis=0)

    # main obs data is in int16  
    return dark_signal_avg.astype(np.int16) 


def make_dark_signal_std_image(dark_l0_path: str):
    """
    Not a pipeline step at the moment. Just for QA. Could be used for new BDE. 
    """
    from astropy.io import fits

    with fits.open(dark_l0_path) as hdul:
        dark_obs_data = hdul[0].data

    # check everything looks normal (it should) 
    bands, rows, cols = dark_obs_data.shape
    print(dark_obs_data.shape)
    if rows <= 4: 
        print("This dark signal obs is probably too short to be useful.")
        
    if bands not in (86, 260) or cols not in (320, 640):
        print("This has the wrong number of channels and/or spatial samples.") 
        
    # this is not listed in the DPSIS or Green but I think it makes sense to
    # exclude the first two frames and the last two when computing the std 
    # dark signal bc sometimes they exhibit odd and anomalous characteristics.
    dark_signal_std = np.std(dark_obs_data.transpose(1, 0, 2)[2:-2, :, :], axis=0)

    return dark_signal_std

    
def bad_detector_element_correction(dss_image: np.ndarray, bde_path: str): 
    """
    Elements are flagged 0-5 in these fits files. Values 1-4 seem to indicate 
    things that are "flagged", 1 being the lowest level and 4 the highest. I think
    the mission interpolated everything flagged 1-4 in the spectral direction. 
    This does seem a little problematic to me because the filter seams (channel features) 
    are flagged, usually as 4. So do we interpolate across them twice? I guess so? 
    """
    from astropy.io import fits 

    with fits.open(bde_path) as hdul:
        bde_map = hdul[0].data  
    
        bad_mask = bde_map != 0  # pixels to interpolate across in the spectral / y direction  
    
        n_frames, H, W = dss_image.shape
    
        for col in range(W):
            col_mask = bad_mask[:, col] 
            if not np.any(col_mask):
                continue
    
            good_rows = np.where(~col_mask)[0]
            if good_rows.size == 0:
                continue  # everything is bad! 
    
            bad_rows = np.where(col_mask)[0]

            # need to investigate different linear interpolation methods for speed 
            # and how they treat edge cases
            for frame in range(n_frames):
                good_vals = dss_image[frame, good_rows, col]
                dss_image[frame, bad_rows, col] = np.interp(
                    bad_rows,
                    good_rows,
                    good_vals,
                ).astype(np.int16) # don't want floats (at least not rn) 
    
        return dss_image


def detector_array_tap_interpolation(obs_data: np.ndarray, cols: list): 
    """
    Columns related to the read-out are linearly interpolated in the spatial 
    direction (columns on either side). The columns are 161, 321, 481 for
    target and 81, 161, 241 for global mode.
    """
    for col in cols:
        left_col = col - 1
        right_col = col + 1
        # we are only going across 1 pixel and not at vertical edges so no need to do 
        # anything complicated 
        obs_data[:, :, col] = (obs_data[:, :, left_col] + obs_data[:, :, right_col]) / 2.0

    return obs_data

def filter_seam_interpolation(obs_data: np.ndarray, channels: list): 
    """
    Order sorting filter seams show up on the detector, we linearly interpolate
    across them in the spectral direction (these are horizontal features). For 
    target mode they are channels 41, 42, and 116 and for global mode they are 
    13 and 50.
    """
    for channel in channels:
        top_channel = channel - 1
        bottom_channel = channel + 1
        # this doesn't work properly for 41/42 right now in target mode 
        obs_data[:, channel, :] = (obs_data[:, top_channel, :] + obs_data[:, bottom_channel, :]) / 2.0
        
    return obs_data 
    
def get_cal_paths(obs: str, local_dir: str): 
    """
    Check spreadsheet containing cal filenames pulled from L1B header files 
    in the PDS3 data. We want to use the latest version available, bc it's
    possible they changed cal files for later versions of the data. 

    Let's also check their validity (ie is this a bad bde / bad dark, how 
    many pixels are flagged, do the cal files exist). 
    """
    cal_files = pd.read_csv(f"{local_dir}/obs_to_cal_file_mapping/{cal_reference}")
    
    cal_files = cal_files[cal_files['obs_id'] == obs.upper()] 

    if len(cal_files) == 0: 
        print("This is not a valid observation with cal file mappings.")  

    # we want the highest version of the observation (ie V03 not V02) 
    cal_files = cal_files.loc[cal_files['version'].str.extract(r'(\d+)')[0].astype(int).idxmax()]

    cal_paths = {}

    data_dir = local_dir+"/data/"
    
    temp = cal_files['obs_temperature']

    mode = "global" if cal_files['obs_type'] == "G" else "target"
    
    beta_angle = cal_files['obs_beta_angle'] 
    
    cal_paths['dark'] = f"{data_dir}/{cal_files['dark_signal_id'].lower()}_l0.fits"
    cal_paths['obs_flat'] = f"{data_dir}/{cal_files['flat_field_id'].lower()}_ff.fits"
    cal_paths['lab_flat'] = f"{local_dir}/cal_files/{lab_flat_fields[cal_files['obs_type']]}"
    cal_paths['bde'] = f"{data_dir}{cal_files['bad_detector_map_id'].lower()}_bde.fits"
    cal_paths['obs'] = f"{data_dir}{cal_files['obs_id'].lower()}_l0.fits"

    # warnings about the cal files, if applicable 
    print(f"Obs {obs} is a {mode} observation conducted at {temp} K and\
     beta angle {beta_angle} during observational period {cal_files['obs_period']}.")

    if cal_files['bad_bde'] or cal_files['bad_dark'] == True: 
        # only applies to a handful of observations, eventually we should 
        # designate alteratives 
        print(f"A dark signal observation used in calibrating this observation was\
              mispointed and should not be used.")

    if not (cal_files['flat_in_usgs'] and 
            cal_files['bde_in_usgs'] and 
            cal_files['dark_in_usgs'] and 
            cal_files['cal_l0_in_usgs']):
        # this only applies to a hanfdul of observations 
        print(f"One of the cal files required is missing from the USGS PDS.") 

    # we may went to return temp and beta angle for cal in the future? 
    for key, filename in cal_paths.items():
        if Path(filename).exists():
            continue 
        else: 
            print(f"{filename} not found in {data_dir}.")
        
    return cal_paths, cal_files['obs_type']

In [ ]:
def l0_to_l1b(obs: str, local_dir: str): 
    """
    String together steps 1-9 eventually. 
    """
    from pathlib import Path

    # make folder for saving output & intermediate steps 
    Path(f"{local_dir}/outputs").mkdir(exist_ok=True)
    
    obs = obs.lower() 
    
    # check file mappings for cal files based on obs number, print obs temp 
    # also check if it's a valid obs we have file mappings for 
    cal_paths, mode = get_cal_paths(obs, local_dir)

    # load obs data and convert to "detector pov" orientation to make cal file 
    # application easier 
    print("loading obs data") 
    with fits.open(cal_paths['obs']) as hdul:
            obs_data = hdul[0].data  
    obs_data = obs_data.transpose(1, 0, 2) 

    ### STEP 1: DARK SIGNAL SUBTRACTION 
    # get average dark signal
    print("subtracting dark signal")
    dark_signal_avg = make_dark_signal_mean_image(cal_paths['dark'])
    
    fits.writeto(
        f"{local_dir}/outputs/{obs}_dark_avg.fits", 
        dark_signal_avg, 
        overwrite=True
    )
    # make dark signal subtracted image (DSS) 
    obs_data -= dark_signal_avg

    fits.writeto(
        f"{local_dir}/outputs/{obs}_after_dss.fits",
        obs_data, 
        overwrite=True
    )

    del dark_signal_avg 

    ### STEP 2: BAD DETECTOR ELEMENT CORRECTION
    # interpolate bad detector elements 
    print("interpolating across bad detector elements") 
    obs_data = bad_detector_element_correction(obs_data, cal_paths['bde'])
    
    fits.writeto(
        f"{local_dir}/outputs/{obs}_after_bde.fits",
        obs_data, 
        overwrite=True
    )

    ### STEP 3: DETECTOR ARRAY TAP INTERPOLATION
    print("running detector array tap interpolation") 

    obs_data = detector_array_tap_interpolation(
        obs_data,
        cal_channel_and_sample_indices['detector_array_tap_columns'][mode]
    ) 

    fits.writeto(
        f"{local_dir}/outputs/{obs}_after_tap_interp.fits",
        obs_data, 
        overwrite=True
    )

    ### STEP 4: FILTER SEAM INTERPOLATION
    print("running filter seam interpolation") 

    obs_data = filter_seam_interpolation(
        obs_data, 
        cal_channel_and_sample_indices['filter_seam_channels'][mode]
    ) 

    fits.writeto(
        f"{local_dir}/outputs/{obs}_after_filter_seam.fits",
        obs_data, 
        overwrite=True
    )

    return obs_data 

In [ ]:
data =  l0_to_l1b("m3g20090125t172601", "/home/bekah/m3-pipeline-dev")

In [ ]:
# trying to improve the way the BDE is applied by using raster math more efficiently 
# 
import matplotlib.pyplot as plt 
import numpy as np 
from astropy.io import fits 

bde_path = "/home/bekah/m3-pipeline-dev/data/m3g20090125t171718_bde.fits"

with fits.open(bde_path) as hdul:
    bde_map = hdul[0].data  
    bad_mask = bde_map != 0 

plt.imshow(bad_mask, cmap='binary') 

In [ ]:
index = np.mgrid[0:86, 0:320] 
index

In [ ]:
bad_col = index[1, :, :][bad_mask] 
bad_row = index[0, :, :][bad_mask]

In [ ]:
good_rows_per_col = index[0, :, :][~bad_mask] 
good_cols_per_col = index[1, :, :][~bad_mask] 

In [ ]:
top_row_index = bad_row + 1 
bottom_row_index = bad_row - 1

In [ ]:
plt.scatter(bad_col, top_row_index, s=.1) 
plt.scatter(bad_col, bottom_row_index, s=.1)
#plt.scatter(good_cols_per_col, good_rows_per_col, s=.2)

In [ ]:
good_rows_per_col[good_cols_per_col == 60]

In [ ]:
bad_row[bad_col == col]

In [ ]:
col = 60 

# subtract all possible row values from bad row values (this will automatically exclude neighboring bad row values) 
options = [good_rows_per_col[good_cols_per_col == col] - i for i in bad_row[bad_col == col]]



In [ ]:
options

In [ ]:
# open BDE 
bde_path = "/home/bekah/m3-pipeline-dev/data/m3g20090125t171718_bde.fits"

with fits.open(bde_path) as hdul:
    bde_map = hdul[0].data  
    bad_mask = bde_map != 0 

# make array with indices of cols and rows 
index = np.mgrid[0:86, 0:320] 

# things we want to interpolate across
bad_col = index[1, :, :][bad_mask] 
bad_row = index[0, :, :][bad_mask]

# pixels we can still use 
good_cols_per_col = index[1, :, :][~bad_mask] 
good_rows_per_col = index[0, :, :][~bad_mask] 

# this part gets column specific
# and we need to treat edge cases still (top and bottom row etc) 
col = 60 
# subtract all possible row values from bad row values (this will automatically exclude neighboring bad row values) 
options = [good_rows_per_col[good_cols_per_col == col] - i for i in bad_row[bad_col == col]]

# correctly finds the indices of the closest pixel option diff 
option_top = [np.where(i > 0, i, np.inf).argmin() for i in options]
option_bottom = [np.where(i < 0, i, -np.inf).argmax() for i in options]

# then map that to the actual correct row values to interpolate across 
top_row = [good_rows_per_col[good_cols_per_col == col][i] for i in option_top] 
bottom_row = [good_rows_per_col[good_cols_per_col == col][i] for i in option_bottom] 

top_row = np.array(top_row)
bottom_row = np.array(bottom_row)

# so 
# this is the row val of the things we want to interpolate bad_row[bad_col == col] 
# this is the top row: top_row 
# this is the bottom row: bottom row 

# so we can do some basic math and then map them back into the image frame by frame, I think 
# and we should calculate the new values using array math not in a loop 

# so then I think factor needs to be saved to an array? and then we can apply the equation to the whole array very quickly? 
# instead of calculating all of the above for every frame. we just need to do it for every column, once. 
factor = (bad_row[bad_col == col] - bottom_row) / (top_row - bottom_row) 
new_val = (top_row - bottom_row) * factor + bottom_row 
new_val = np.round(new_val).astype(np.int32) 

In [ ]:
new_val

In [ ]:
bad_row[bad_col == col]

In [ ]:
top_row

In [ ]:
bottom_row

In [ ]:
top_row = [good_rows_per_col[good_cols_per_col == col][i] for i in option_top] 
bottom_row = [good_rows_per_col[good_cols_per_col == col][i] for i in option] 

In [ ]:
top_row # row values of good top rows for each flagged pixel 

In [ ]:
bottom_row

In [ ]:
good_rows_per_col[good_cols_per_col == col][12]

In [ ]:
options[0][0]

In [ ]:
options[1][0]

In [ ]:
min_option = max([x for sub in options for x in sub if x < 0])


In [ ]:
min_option

In [ ]:
top_row = bad_row

In [ ]:
# so now I have all col, row pairs for the nominal top and bottom pixel for interpolation
# but this doesn't account for "runs" of bad pairs where there could be several in a row that are bad 
# I need a clever way to accomodate for those? 

# so maybe I need to keep masking the coordinates until none are bad? 
# or make one array at the beginning showing the closest not bad row value. I think that is probably best because then it only has 
# to run once each time 

with fits.open(bde_path) as hdul:
        bde_map = hdul[0].data  
    
        bad_mask = bde_map != 0  # pixels to interpolate across in the spectral / y direction  
    
n_frames, H, W = dss_image.shape

top_pixel = np.array(H, W)
bottom_pixel = np.array(H, W) 
    
for col in range(W):
    last_good_pixel = 0 
    for row in range(H): 
        if bad_mask == 0: 
            top_pixel[col, row] = row 
            bottom_pixel[col, row] = row 
            last_good_pixel = row 
        if bad_mask == 1: 
            top_pixel[col, row] =  last_good_pixel 

# pull indices of "good" rows 
# for each row, subtract all row values. the closest top one will be the smallest positive integer and the closest bottom one will have 
# the smallest negative integer 

In [ ]:
for col in range(W):
    last_good_pixel = 0 
    for row in range(H): 
        if bad_mask == 0: 
            top_pixel[col, row] = row 
            bottom_pixel[col, row] = row 
            last_good_pixel = row 
        if bad_mask == 1: 
            top_pixel[col, row] =  last_good_pixel 

In [ ]:
import py.ma as ma 
.compressed() 

In [ ]:
with fits.open(bde_path) as hdul:
        bde_map = hdul[0].data  
        bad_mask = bde_map != 0  # pixels to interpolate across in the spectral / y direction  

n_frames, H, W = bad_mask.shape

for col in range(W):
    bad_mask[col, :]
